- Read QuakeML
- Filter catalog
- Define template parameters
- Download miniseeds surrounding the templates
- Create template objects from miniseeds, save
- Perform detection using template matching

In [2]:
import obspy
import eqcorrscan
import numpy as np
import pandas as pd
import tarfile

### Define network parameters

In [2]:
stations = ['KEMF','NCHR','ENWF','KEMO']
network = 'NV'

### Define template parameters

In [3]:
lowcut = 8.0
highcut = 35.0
filt_order = 4
data_pad = 20.
samp_rate = 200.0
length = 0.5
prepick = 0.05
process_len = 86400
min_snr = 0.1
swin = 'all'

### Define starting catalog filters

In [4]:
min_lat = 47.9
max_lat = 48.05
min_lon = -129.15
max_lon = -129.05
min_time = obspy.UTCDateTime(2017,6,1,1,30,0)
max_time = obspy.UTCDateTime(2017,7,1,1,30,0)
min_magnitude = 1.2
max_magnitude = 'NaN'

### Define detection parameters

In [5]:
threshold = 0.6
threshold_type = 'av_chan_corr'
trig_int = 1
# etc.

### Define pathnames

In [6]:
# Path to starting catalog in QuakeML format:
starting_cat_path = 'endquakes_2017.xml'
# Path to miniseed waveform data in tar format:
data_path = "/data/wsd01/endeavor.tar.gz"

### Read in starting earthquake catalog, in QuakeML format

In [7]:
cat = obspy.core.event.catalog.read_events(starting_cat_path)

#### Filter catalog

In [8]:
min_time = min_time.datetime.strftime('%Y-%m-%dT%H:%M:%S')
max_time = max_time.datetime.strftime('%Y-%m-%dT%H:%M:%S')
t1_filter = "time > " + min_time
t2_filter = "time < " + max_time
lon1_filter = "longitude > " + str(min_lon)
lon2_filter = "longitude < " + str(max_lon)
lat1_filter = "latitude > " + str(min_lat)
lat2_filter = "latitude < " + str(max_lat)
mag1_filter = "magnitude > " + str(min_magnitude)
mag2_filter = "magnitude < " + str(max_magnitude)
cat = cat.filter(t1_filter,t2_filter,lon1_filter,lon2_filter,lat1_filter,lat2_filter,mag1_filter)

In [9]:
# Add in a phase hint to the picks
for event in cat.events:
    for pick in event.picks:
        pick_id = pick.resource_id
        arr = [a for a in event.origins[0].arrivals if a.pick_id==pick_id]
        pick.phase_hint=arr[0].phase

#### Optional: download waveform data for the catalog duration

In [6]:
# Get list of stations and channels from QuakeML
picks = [p.picks for p in cat.events]
picks = sum(picks,[])

networks = [p.waveform_id.network_code for p in picks]
stations = [p.waveform_id.station_code for p in picks]
channels = [p.waveform_id.channel_code[0:2] + '*' for p in picks]
# Toss pressure channels:
channelToRemove = 'HD*'
channels = [value for value in channels if value != channelToRemove]

sta_list = [f"{n}.{s}..{c[0:2]}" for n, s, c in zip(networks,stations,channels)]
sta_list = np.unique(sta_list)

# Get all the info for those stations from IRIS
network = ",".join((np.unique(networks)).tolist())
channel = ",".join((np.unique(channels)).tolist())
station = ",".join((np.unique(stations)).tolist())


## Miniseed naming conventions:
KEMF.NV.2017.yearday

In [14]:
# Loop over days and download each (unprocessed) into a directory
path = 'data_june2017/'
t1 = obspy.UTCDateTime(2017,6,1)
t2 = obspy.UTCDateTime(2017,6,3)

client = obspy.clients.fdsn.client.Client('IRIS')

time_bins = np.arange(t1.datetime,t2.datetime,pd.Timedelta(1,'days'))
time_bins = [pd.to_datetime(t) for t in time_bins]

for t in time_bins:
    
    t1 = obspy.UTCDateTime(t)
    t2 = obspy.UTCDateTime(t + pd.Timedelta(1,'day'))
    
    pathname = path + t1.strftime("%Y%m%d")+'.mseed'
    
    st = client.get_waveforms(network,station,'',channel,t1,t2)
    
    st.write(pathname)

## Link to the already downloaded data

In [7]:
tar = tarfile.open(data_path, "r")

## Create templates from local miniseed data

In [ ]:
template_list = []

# Start base day as none
base_day = None

# Loop over events in catalog and make a template for each
# Load one day-long stream in as needed
for i,ev in enumerate(cat):
    
    # Check if we need to pull in a new stream
    if ev.origins[0].time.datetime.strftime('%Y%m%d') != base_day:
        # Reset the day we are on and pull in a new stream
        base_day = ev.origins[0].time.datetime.strftime('%Y%m%d')
        print('Downloading templates for ' + base_day)
        year = str(ev.origins[0].time.datetime.year)
        yearday = ev.origins[0].time.datetime.strftime("%j")
        # Loop through stations to make the complete stream
        st = obspy.core.stream.Stream()
        for sta in stations:
            path = 'data/' + str(network) + '/' + year + '/' + yearday + '/' + sta + '.' + network + '.' + year + '.' + yearday
            tfile = tar.extractfile(path)
            stream = obspy.core.stream.read(tfile)
            st += stream
        st.resample(samp_rate)
    
    temp_cat = obspy.core.event.Catalog(events=[ev])

    # Process stream
    template_st = eqcorrscan.core.template_gen.template_gen(method="from_meta_file", st=st,lowcut=lowcut, highcut=highcut, samp_rate=samp_rate, length=length,
    filt_order=filt_order, prepick=prepick,meta_file=temp_cat, data_pad=data_pad,
    process_len=process_len, min_snr=min_snr, parallel=False,swin=swin,delayed=True,skip_short_chans=True)

    # Make template from processed waveform
    template = eqcorrscan.core.match_filter.template.Template(name=str(ev.resource_id.id)[-6:], st=template_st[0], lowcut=lowcut, highcut=highcut, samp_rate=samp_rate, filt_order=filt_order, process_length=process_len, prepick=prepick, event=ev)
    template_list.append(template)

In [23]:
# Make a "tribe" from the templates
# Make sure that resource id in the tribe events matches the template
tribe = eqcorrscan.core.match_filter.tribe.Tribe(templates=template_list)
for i,t in enumerate(tribe):
    tribe[i].event.origins[0].resource_id = t.name

In [ ]:
# Write templates to file, if desired
tribe.write('sep2017_templates')

## Perform detection on local miniseed data

In [8]:
# Read in template tribe if one is already saved
tribe = eqcorrscan.core.match_filter.tribe.Tribe().read('sep2017_templates.tgz')

### Unfortunately, there are lots of gaps in this data (see below). We will have to skip those gaps unless we make the process length of the templates shorter, at 3600 s.
http://service.iris.edu/fdsnws/availability/1/query?network=NV&station=KEMF&channel=EHZ&start=2017-06-01T00:00:00&end=2017-07-01T00:00:00

In [29]:
t1 = obspy.UTCDateTime(2017,6,4)
t2 = obspy.UTCDateTime(2017,6,10)
time_bins = np.arange(t1.datetime,t2.datetime,pd.Timedelta(1,'days'))
time_bins = [pd.to_datetime(t) for t in time_bins]

# Loop over days, with one party of detections for each day
parties = []
for t in time_bins:
    t1 = obspy.UTCDateTime(t)
    t2 = obspy.UTCDateTime(t + pd.Timedelta(1,'day'))
    
     ###### Retrieve stream #####
    
    # Get year and year day
    year = str(t1.datetime.year)
    yearday = t1.datetime.strftime("%j")
    # Loop through stations
    st = obspy.core.stream.Stream()
    for sta in stations:
        path = 'data/' + str(network) + '/' + year + '/' + yearday + '/' + sta + '.' + network + '.' + year + '.' + yearday
        tfile = tar.extractfile(path)
        stream = obspy.core.stream.read(tfile)
        st += stream
    st.resample(samp_rate)
    # Catch for short streams due to data gaps
    if len(st[0])/st[0].stats.sampling_rate < process_len:
        print(len(st[0])/st[0].stats.sampling_rate)
        print('Data is too short for day ' + yearday + ', skipping detection')
        continue
    print("Data pulled in for day " + yearday + ", starting detection")

    stream = st
    party = tribe.detect(stream, threshold=threshold, threshold_type=threshold_type,trig_int=trig_int,parallel_process=False)
    print("Detection finished for day " + yearday)
    parties.append(party)

Data pulled in for day 155, starting detection
Detection finished for day 155
28054.315
Data is too short for day 156, skipping detection
14654.47
Data is too short for day 157, skipping detection
22484.785
Data is too short for day 158, skipping detection
43008.55
Data is too short for day 159, skipping detection
40285.16
Data is too short for day 160, skipping detection


In [54]:
# Combine parties from separate days
families = []
num_templates = len(parties[0].families)
for i in range(num_templates):
    fam_list = [p[i] for p in parties]
    fam = fam_list[0]
    for f in fam_list[1:]:
        if f is not None:
            fam = fam.append(f)
    families.append(fam)
party = eqcorrscan.core.match_filter.party.Party(families=families)

### Plot up some of the detections!

In [ ]:
client = obspy.clients.fdsn.Client('IRIS')
import matplotlib.pyplot as plt

for d in party[0]:

    print(d.detect_val)
    picks = [obspy.UTCDateTime(i.time.datetime) for i in d.event.picks]

    t1 = min(picks)-5
    t2 = max(picks)+5
    stations = 'KEMO,ENWF,NCHR,KEMF'
    chans = 'EH*,HH*'
    st = client.get_waveforms('NV',stations,'*',chans,t1,t2)
    st.filter('bandpass',freqmin=8,freqmax=35)
    st = st.taper(max_percentage=0.5)
    #st = st.sort()
    st_stations = [tr.stats.station for tr in st]
    st_chans = ['HHZ','HHN','HHE','EHZ','EHN','EHE','EHZ','EHN','EHE','EHZ','EHN','EHE']
    fig = st.plot(handle=True,size=[800,600]);

    for p in d.event.picks:
        sta = p.waveform_id.station_code
        chan = p.waveform_id.channel_code
        ind1 = [i for i,li in enumerate(st_stations) if li==sta]
        ind2 = [i for i,li in enumerate(st_chans) if li==chan]
        ind = [value for value in ind1 if value in ind2]
        fig.get_axes()[ind[0]].vlines(x=p.time.datetime,ymin=-3000,ymax=3000,colors ='r',linewidth=1)

    plt.show()

## Write detections as a QuakeML file

In [64]:
party.write('test_quakeml.xml', format='quakeml')

Writing only the catalog component, metadata will not be preserved


Party of 77 Families.

## We can see that the written catalog contains picks for each event, and cross correlation information as well.

In [3]:
det_cat = obspy.core.event.catalog.read_events('test_quakeml.xml')

In [6]:
det_cat[0].picks

[Pick(resource_id=ResourceIdentifier(id="smi:local/13408b5a-d11c-4031-9bff-7e71dccc7631"), time=UTCDateTime(2017, 6, 1, 20, 2, 26, 470000), waveform_id=WaveformStreamID(network_code='NV', station_code='ENWF', channel_code='HHE', location_code='')),
 Pick(resource_id=ResourceIdentifier(id="smi:local/7b7a9cca-7aed-4f2d-9629-4776d6324b76"), time=UTCDateTime(2017, 6, 1, 20, 2, 24, 980000), waveform_id=WaveformStreamID(network_code='NV', station_code='ENWF', channel_code='HHZ', location_code='')),
 Pick(resource_id=ResourceIdentifier(id="smi:local/5e674936-cf4f-40b8-87de-5f28fa228849"), time=UTCDateTime(2017, 6, 1, 20, 2, 26, 500000), waveform_id=WaveformStreamID(network_code='NV', station_code='KEMF', channel_code='EHE', location_code='')),
 Pick(resource_id=ResourceIdentifier(id="smi:local/a06cab98-5705-4a8b-8bec-84e523dc75e5"), time=UTCDateTime(2017, 6, 1, 20, 2, 25, 75000), waveform_id=WaveformStreamID(network_code='NV', station_code='KEMF', channel_code='EHZ', location_code='')),
 Pick

In [7]:
det_cat[0].comments

[Comment(text='Template: 515146', resource_id=ResourceIdentifier(id="smi:local/e7438cfe-2c59-4827-9356-c88ac4b230cb")),
 Comment(text='threshold=3.5999999046325684', resource_id=ResourceIdentifier(id="smi:local/3c055301-d8a7-449f-af36-cff6eba3df8e")),
 Comment(text='detect_val=4.058650016784668', resource_id=ResourceIdentifier(id="smi:local/1990cc75-6b13-48f6-90e7-161cee6ad005")),
 Comment(text="channels used: ('ENWF', 'HHE') ('ENWF', 'HHZ') ('KEMF', 'EHE') ('KEMF', 'EHZ') ('NCHR', 'EHE') ('NCHR', 'EHZ')", resource_id=ResourceIdentifier(id="smi:local/e8b2a33e-316e-4f9d-90f9-75adfb69a8c3"))]

### Optional To-Do: post-processing code to filter out repeat detections